# Retrieval-Augmented Generation (RAG) Pipeline

## 1. Installation

In [ ]:
%pip install -q -U ragas bitsandbytes transformers accelerate langchain chromadb \
    langchain-huggingface datasets python-dotenv langchain-text-splitters langchain-community \
    sentence-transformers

## 2. Imports

In [ ]:
import os
import torch
from datasets import load_dataset, Dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from sentence_transformers import CrossEncoder
from dotenv import load_dotenv
import pandas as pd

## 3. Data Loading & Preprocessing
We use the IBM WatsonX documentation dataset.

In [ ]:
# Load the IBM corpus dataset
corpus_dataset = load_dataset("ibm-research/watsonxDocsQA", "corpus", split="train")
print(type(corpus_dataset[0]))

# Convert into LangChain Document objects
documents = []
source_dict = {}
for entry in corpus_dataset:
    content = f"Title: {entry['title']}\n\n{entry['document']}"
    doc = Document(page_content=content, metadata={"source": entry['doc_id']})
    source_dict[entry['doc_id']] = content
    documents.append(doc)

print(f"Loaded {len(documents)} documents.")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=300)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} chunks.")

## 4. Vector Database Construction
Using sentence-transformers to create embeddings and ChromaDB for storage.

In [ ]:
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Process in batches to manage memory
batch_size = 100
vectorstore = None

print(f"Starting embedding for {len(docs)} chunks...")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(documents=batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)
    print(f"✅ Completed {i + len(batch)} / {len(docs)} chunks...")

print("✅ Vector store ready.")

## 5. Load Mistral-7B-Instruct (4-bit quantized)

In [ ]:
# Load HuggingFace token from .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print(f"✅ Model loaded on device: {model.device}")

## 6. Retriever & Reranker Setup

In [ ]:
# MMR retriever — fetches a diverse pool, reranker picks the best
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 15,          # final candidates passed to reranker
        "fetch_k": 50,    # initial pool fetched via similarity search
        "lambda_mult": 0.6  # 1.0 = pure similarity, 0.0 = max diversity
    }
)

# Cross-encoder reranker — more accurate than bi-encoder for final scoring
rerank_model = CrossEncoder('BAAI/bge-reranker-base', device="cpu")

print("✅ Retriever and reranker ready.")

## 7. RAG Pipeline

In [ ]:
def rag_pipeline(question):
    initial_docs = retriever.invoke(question)
    if not initial_docs:
        return "No relevant documents found.", []

    pairs = [[question, doc.page_content] for doc in initial_docs]
    scores = rerank_model.predict(pairs)
    scored_docs = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in scored_docs[:3]]
    print(top_docs,"top 3 docs")
    # context = "\n\n".join([doc.page_content for doc in top_docs])
    context_parts = []
    for doc in top_docs:
        source_id = doc.metadata.get("source", "Unknown")
        # Prepend the source ID to the chunk so the LLM knows where it came from
        context_parts.append(f"[Source: {source_id}]\n{doc.page_content}")
        
    context = "\n\n".join(context_parts)
    # messages = [
    #     {
    #         "role": "user",
    #         "content": f"Context: {context}\n\nQuestion: {question}\n\nAnswer using only the provided context."
    #     }
    # ]
    # messages = [
    #         {
    #             "role": "user",
    #             "content": f"""
    #     You are a retrieval-based assistant.

    #     Use ONLY the provided context to answer the question.

    #     Rules:
    #     - Do NOT use prior knowledge.
    #     - Do NOT infer beyond explicit statements.
    #     - Do not infer associations unless explicitly stated.
    #     - If the answer is not clearly available in the context, say:
    #     "The provided context does not contain enough information to answer this question."
    #     - Be concise and factual.

    #     Context:
    #     {context}

    #     Question:
    #     {question}
    #     """
    #         }
    #     ]
    messages = [
        {
        "role": "system",
        "content": "You are a strict RAG system. You ONLY use explicitly stated context. No external knowledge. You MUST cite your sources for every factual claim you make using the provided [Source: ID] tags."
        },
        {
        "role": "user",
        "content": f"""
        Step 1: Extract exact relevant lines from context.
        Step 2: If none exist, say "Insufficient context."
        Step 3: Answer ONLY using extracted lines, be concise and factual.
        Step 4: At the end of every sentence or distinct claim, append the corresponding source ID exactly as formatted in the context (e.g., [Source: 8109B6380...]).
        Step 5: Do not combine multiple components into a higher-level product definition unless explicitly stated in context.
        Context:
        {context}

        Question:
        {question}
        """
        }
    ]
    

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True        
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,                            
            max_new_tokens=400,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    
    prompt_length = inputs["input_ids"].shape[-1]
    answer = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return answer, top_docs

## 8. Quick Test

In [ ]:
question = "What foundation models have been built by IBM?"
print(f"Question: {question}")
print("-" * 30)
answer, used_docs = rag_pipeline(question)
print(f"Answer: {answer}")
print(f"\n({len(used_docs)} chunks used by the LLM)")

## 9. Load Test Dataset

In [ ]:
# FIX: corrected typo from 'datset_test' → 'dataset_test'
# Use the test split for proper evaluation
from datasets import load_dataset

dataset_test = load_dataset(
    "ibm-research/watsonxDocsQA",
    "question_answers",
    split="test"
)
print(dataset_test[0])
print(f"Total test samples: {len(dataset_test)}")


## 10. Retrieval Accuracy: Hit@1 and Hit@3 with Reranker

In [ ]:
def check_retrieval(question, true_id, k=3):
    docs = retriever.invoke(question)

    if not docs:
        print(f"No results for: {question[:40]}")
        return False, False

    pairs = [[question, d.page_content] for d in docs]
    scores = rerank_model.predict(pairs)

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    top_docs = [d for _, d in ranked[:k]]

    hit_at_1 = False
    hit_at_3 = False

    print("\n------------------------------")
    print("Q:", question)
    print("Expected:", true_id)
    print("------------------------------")

    true_id = str(true_id).strip()

    for idx, doc in enumerate(top_docs):
        pred_id = str(doc.metadata.get("source")).strip()

        match = pred_id == true_id

        if match:
            if idx == 0:
                hit_at_1 = True
            hit_at_3 = True

        print(f"{idx+1}. {pred_id} | match={match}")

    return hit_at_1, hit_at_3


hits1, hits3 = [], []

print("running eval...\n")

for i, row in enumerate(dataset_test.select(range(10))):
    q = row["question"]
    target = row["correct_answer_document_ids"]

    h1, h3 = check_retrieval(q, target, k=3)

    hits1.append(h1)
    hits3.append(h3)

    if (i + 1) % 10 == 0:
        print(f"processed {i+1} samples...")


h1_score = sum(hits1) / len(hits1)
h3_score = sum(hits3) / len(hits3)

print("\n====================")
print("FINAL RESULTS")
print("====================")
print("Hit@1:", round(h1_score, 3))
print("Hit@3:", round(h3_score, 3))

if h1_score > 0.85:
    print("\nlooks good — top result seems reliable enough")
    print("you can probably pass full doc to the LLM")

## 11. Build RAGAS Evaluation Dataset

We now call `rag_pipeline()` once per question and unpack both
the answer and the exact chunks used — no more double retrieval.

In [ ]:
questions = []
answers = []
contexts = []
ground_truths = []

print("Generating responses for RAGAS evaluation...")

for row in dataset_test.select(range(20)):
    question = row["question"]

    # FIX: single call — answer + the exact reranked docs the LLM used
    response, top_docs = rag_pipeline(question)

    questions.append(question)
    answers.append(response)
    contexts.append([doc.page_content for doc in top_docs])  # actual docs used
    ground_truths.append(row["correct_answer"])

    print(f"Done: {question[:60]}...")

# FIX: use 'reference' key (required by RAGAS >= 0.1.9)
# If you're on an older version and get a KeyError, change back to 'ground_truth'
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "reference": ground_truths,
}

eval_dataset = Dataset.from_dict(data)
print(f"\neval_dataset created with {len(eval_dataset)} rows.")
for i in range(len(eval_dataset)):
    print(eval_dataset[i])

In [ ]:
# import os
# import asyncio
# from typing import List, Optional, Any
# from langchain_groq import ChatGroq
# from langchain_core.outputs import ChatResult
# from langchain_core.messages import BaseMessage
# from ragas.llms import LangchainLLMWrapper
# from ragas import evaluate
# from ragas.metrics import (
#     Faithfulness, 
#     AnswerRelevancy, 
#     ContextPrecision, 
#     ContextRecall, 
#     AnswerCorrectness
# )

# class RagasChatGroq(ChatGroq):
#     def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs: Any) -> ChatResult:
#         n = kwargs.pop("n", 1)
#         kwargs["n"] = 1 
#         generations = []
#         for _ in range(n):
#             result = super()._generate(messages, stop, run_manager, **kwargs)
#             generations.extend(result.generations)
#         return ChatResult(generations=generations)

#     async def _agenerate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs: Any) -> ChatResult:
#         n = kwargs.pop("n", 1)
#         kwargs["n"] = 1
#         tasks = [super()._agenerate(messages, stop, run_manager, **kwargs) for _ in range(n)]
#         results = await asyncio.gather(*tasks)
#         generations = []
#         for res in results:
#             generations.extend(res.generations)
#         return ChatResult(generations=generations)
# # =====================================================================

# # 1. Initialize the Groq LLM
# os.environ["GROQ_API_KEY"] = "GROQ_PAI KEY" 

# # FIX 1 & 2: Use the new wrapper above and update the model name
# raw_llm = RagasChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
# evaluator_llm = LangchainLLMWrapper(raw_llm)

# # 2. Initialize metrics and EXPLICITLY pass the Groq LLM to each one
# faithfulness = Faithfulness(llm=evaluator_llm)
# answer_relevancy = AnswerRelevancy(llm=evaluator_llm)
# context_precision = ContextPrecision(llm=evaluator_llm)
# context_recall = ContextRecall(llm=evaluator_llm)

# # AnswerCorrectness also needs the embeddings you created earlier
# answer_correctness = AnswerCorrectness(
#     llm=evaluator_llm, 
#     embeddings=embeddings # Ensure 'embeddings' is defined earlier in your script/notebook
# )

# # 3. Execute the evaluation
# results = evaluate(
#     dataset=eval_dataset, # Ensure 'eval_dataset' is defined earlier in your script/notebook
#     metrics=[
#         faithfulness, 
#         answer_relevancy, 
#         context_precision, 
#         context_recall, 
#         answer_correctness
#     ],
#     llm=evaluator_llm,
#     embeddings=embeddings
# )

# print("\nEvaluation Results:")
# print(results)